In [52]:
import sys
from pathlib import Path
from copy import deepcopy
import pandas as pd
import numpy as np

axiom_utils = Path('/kaggle/input/datasets/harshit1234g/axiomlm-utils')
sys.path.append(str(axiom_utils))
from llm_components import load_sp_tokenizer

In [53]:
dolly_path = axiom_utils / 'dolly_15k'
ds_path = dolly_path / 'databricks-dolly-15k.csv'
tokenizer_path = str(axiom_utils / 'sp_tokenizer.model')

In [54]:
tokenizer = load_sp_tokenizer(tokenizer_path)

In [55]:
df = pd.read_csv(
    filepath_or_buffer= ds_path,
    usecols= [0, 1, 2]
)

In [56]:
df.head()

,instruction,context,response
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...
1,Which is a species of fish? Tope or Rope,NaN,Tope
2,Why can camels survive for long without water?,NaN,Camels use the fat in their humps to keep them...
3,"Alice's parents have three daughters: Amy, Jes...",NaN,The name of the third daughter is Alice
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981."


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15015 entries, 0 to 15014
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   instruction  15015 non-null  object
 1   context      4597 non-null   object
 2   response     15015 non-null  object
dtypes: object(3)
memory usage: 352.0+ KB


In [58]:
def format_instruction(row):
    if row['context'] is np.nan or row['context'] is pd.NA:
        return f'# Instruction:\n{row["instruction"]}\n# Response:\n{row["response"]}'
    return f'# Instruction:\n{row["instruction"]}\n# Context:\n{row["context"]}\n# Response:\n{row["response"]}'

In [59]:
df = df.apply(format_instruction, axis= 1)

In [60]:
data = df.to_numpy()

In [61]:
for i in range(5):
    print('-' * 30)
    print(data[i])
    print()

------------------------------
# Instruction:
When did Virgin Australia start operating?
# Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.[3] It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.[4]
# Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.

------------------------------
# Instruction:
Which is a species of fish? Tope or Rope
# Response:
Tope

------------------------------
# Instruction:
Why can camels survive for long without water?
# Response:
Camels use the fat in their humps to keep them filled

In [62]:
tokens = tokenizer.encode(data.tolist(), out_type= int)

In [63]:
len(tokens)

15015

In [64]:
tokens = [t for t in tokens if len(t) <= 509]
# selecting instances with tokens less than 509
# used 509 so that after adding <eos>, <bos>, and a <pad> the tokens will be 512

In [65]:
len(tokens)

13810

In [66]:
eos_id = tokenizer.eos_id()
bos_id = tokenizer.bos_id()
ignore_idx = -100

In [67]:
ds = np.array([
    np.array([bos_id] + t + [eos_id, ignore_idx], dtype= np.int16) 
    for t in deepcopy(tokens)
], dtype= 'object')

In [68]:
X = np.array([x[:-1] for x in deepcopy(ds)], dtype= 'object')
y = np.array([x[1:] for x in deepcopy(ds)], dtype= 'object')

In [69]:
n = 5
X[n], y[n]

(array([    1,    39,  5109,  1622, 15984,    14, 15951, 15929,   346,
          564,   687,  4626,   379,   264,   647,   285,   353,   288,
          365,   430, 15935,   564,   346,  1405,    67,    14,    39,
          988,  7600, 15984,    14, 15943,  1614,   365,   430,   376,
          262,  4210,   287, 13357,   813,   264,  1701,  2876,  1314,
          390,   376,   293,  2727,   376,   474,   287,  6894,   292,
          589,   928,  3934,  2727, 15936,   465,   288,   365,   430,
         4412,   287,   262,  3118, 15936,  1655,   264,   795, 15930,
          451, 15935,   353,   288,   365,   430,   376,   262,   536,
        10439,   361,   764, 13774,   264,  1701,   359,   264, 15454,
         1703,   293,  3118,   264,   738,  2289,   749,  7809, 15936,
        15993, 15949, 15992,   455,   687,  3329,  3856, 15935,   353,
          288,   365,   430,   376,  1362,   400,   283,   265, 15935,
         2962,  2718,   264,   836,   285,   262,  1912,   646,   300,
      

In [70]:
hash_response = tokenizer.encode('# Response:', out_type= int)
hash_response

[39, 363, 1298, 2898, 15984]

In [71]:
def find_subarray_index(arr, target):
    first_match_indices = np.where(arr == target[0])[0]
    for start_index in first_match_indices:
        end_index = start_index + len(target)
        if np.all(arr[start_index:end_index] == target):
            return end_index

In [72]:
for idx in range(y.shape[0]):
    y[idx][:find_subarray_index(y[idx], hash_response)] = ignore_idx

In [73]:
X[n], y[n]

(array([    1,    39,  5109,  1622, 15984,    14, 15951, 15929,   346,
          564,   687,  4626,   379,   264,   647,   285,   353,   288,
          365,   430, 15935,   564,   346,  1405,    67,    14,    39,
          988,  7600, 15984,    14, 15943,  1614,   365,   430,   376,
          262,  4210,   287, 13357,   813,   264,  1701,  2876,  1314,
          390,   376,   293,  2727,   376,   474,   287,  6894,   292,
          589,   928,  3934,  2727, 15936,   465,   288,   365,   430,
         4412,   287,   262,  3118, 15936,  1655,   264,   795, 15930,
          451, 15935,   353,   288,   365,   430,   376,   262,   536,
        10439,   361,   764, 13774,   264,  1701,   359,   264, 15454,
         1703,   293,  3118,   264,   738,  2289,   749,  7809, 15936,
        15993, 15949, 15992,   455,   687,  3329,  3856, 15935,   353,
          288,   365,   430,   376,  1362,   400,   283,   265, 15935,
         2962,  2718,   264,   836,   285,   262,  1912,   646,   300,
      

In [74]:
np.save('processed_features.npy', X)
np.save('processed_labels.npy', y)